# Flint/GMP/MPFR Fresh Install on Linux

Scripts I use to get flint running. Reach out on Twitter @murage_kibicho for help

In [1]:
%%writefile Download.sh
#!/usr/bin/env bash

# See http://www.flintlib.org/downloads.html
export FLINT_VERSION=3.4.0

# See https://www.mpfr.org/mpfr-current/#download
export MPFR_VERSION=4.2.2

# See https://www.mpfr.org/mpfr-current/#download
export GMP_VERSION=6.3.0


export BUILD=`pwd`/build
export PREFIX=$BUILD/local

# Create the build directory
mkdir -p $BUILD

set -ev
cd $BUILD
### Download and build GMP
curl https://gmplib.org/download/gmp/gmp-$GMP_VERSION.tar.xz -o gmp-$GMP_VERSION.tar.xz
tar xvf gmp-$GMP_VERSION.tar.xz
cd gmp-$GMP_VERSION
./configure --prefix=$PREFIX
make -j$(nproc)
make install

### Download and build MPFR
cd $BUILD
curl https://www.mpfr.org/mpfr-current/mpfr-$MPFR_VERSION.tar.xz -o mpfr-$MPFR_VERSION.tar.xz
tar xvf mpfr-$MPFR_VERSION.tar.xz
cd mpfr-$MPFR_VERSION
./configure --prefix=$PREFIX --with-gmp=$PREFIX
make -j$(nproc)
make install

### Download and build Flint
cd $BUILD
curl https://flintlib.org/download/flint-$FLINT_VERSION.tar.gz -o flint-$FLINT_VERSION.tar.gz
tar xvf flint-$FLINT_VERSION.tar.gz
cd flint-$FLINT_VERSION
./configure --prefix=$PREFIX --with-gmp=$PREFIX --with-mpfr=$PREFIX
make -j$(nproc)
make install

### Update shell config to set PATH permanently
SHELL_RC=""
if [[ "$SHELL" == */bash ]]; then
    SHELL_RC="$HOME/.bashrc"
elif [[ "$SHELL" == */zsh ]]; then
    SHELL_RC="$HOME/.zshrc"
else
    echo "Unknown shell. Please manually add $PREFIX/bin to your PATH."
fi

if [[ -n "$SHELL_RC" ]]; then
    # Only add the lines if they aren't already present
    grep -qxF "export PATH=\"$PREFIX/bin:\$PATH\"" $SHELL_RC || echo "export PATH=\"$PREFIX/bin:\$PATH\"" >> $SHELL_RC
    grep -qxF "export LD_LIBRARY_PATH=\"$PREFIX/lib:\$LD_LIBRARY_PATH\"" $SHELL_RC || echo "export LD_LIBRARY_PATH=\"$PREFIX/lib:\$LD_LIBRARY_PATH\"" >> $SHELL_RC
    grep -qxF "export PKG_CONFIG_PATH=\"$PREFIX/lib/pkgconfig:\$PKG_CONFIG_PATH\"" $SHELL_RC || echo "export PKG_CONFIG_PATH=\"$PREFIX/lib/pkgconfig:\$PKG_CONFIG_PATH\"" >> $SHELL_RC

    echo "Paths added to $SHELL_RC. Run 'source $SHELL_RC' to apply them now."
fi





Writing Download.sh


In [2]:
!chmod +x Download.sh
!./Download.sh
!source ~/.bashrc 2>/dev/null
!source ~/.zshrc 2>/dev/null

Streaming output truncated to the last 5000 lines.
  CC  nfloat/mat.c
  CC  nfloat/mat_mul.c
  CC  nfloat/nfixed.c
  CC  nfloat/nfloat.c
  CC  mpn_extras/debug.c
  CC  mpn_extras/divides.c
  CC  mpn_extras/divrem_preinv1.c
  CC  mpn_extras/divrem_preinvn.c
  CC  mpn_extras/factor_trial.c
  CC  mpn_extras/factor_trial_tree.c
  CC  mpn_extras/fmms1.c
  CC  mpn_extras/gcd_full.c
  CC  mpn_extras/get_d.c
  CC  mpn_extras/get_str.c
  CC  mpn_extras/inlines.c
  CC  mpn_extras/mod_preinvn.c
  CC  mpn_extras/mul_basecase.c
  CC  mpn_extras/mul.c
  CC  mpn_extras/mulhigh_basecase.c
  CC  mpn_extras/mulhigh.c
  CC  mpn_extras/mulhigh_naive.c
  CC  mpn_extras/mulhigh_recursive.c
  CC  mpn_extras/mullow.c
  CC  mpn_extras/mulmod_2expp1_basecase.c
  CC  mpn_extras/mulmod_precond.c
  CC  mpn_extras/mulmod_precond_shoup.c
  CC  mpn_extras/mulmod_preinv1.c
  CC  mpn_extras/mulmod_preinvn.c
  CC  mpn_extras/mul_toom22.c
  CC  mpn_extras/mul_toom32.c
  CC  mpn_extras/preinv1.c
  CC  mpn_extras/preinvn.c

##Set Python local variables

In [3]:
import os

PREFIX = "/content/build/local"

os.environ["PATH"] = f"{PREFIX}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{PREFIX}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PKG_CONFIG_PATH"] = f"{PREFIX}/lib/pkgconfig:" + os.environ.get("PKG_CONFIG_PATH", "")


In [4]:
%%writefile main.c
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <stdlib.h>
#include <assert.h>
#include <stdbool.h>
#include <time.h>
#include <math.h>
#include <gmp.h>
#include <time.h>
#include <flint/flint.h>
#include <flint/fmpz.h>
#include <flint/fmpzi.h>
#include <flint/fmpq.h>
#include <flint/fmpz_factor.h>

int main()
{
    printf("Install complete\n");
    gmp_printf("Hello fron GMP\n");
    return 0;
}

Writing main.c


In [5]:
!clear && gcc main.c -o m.o \
  -I/content/build/local/include \
  -L/content/build/local/lib \
  -lflint -lmpfr -lgmp -lm \
  -Wl,-rpath,/content/build/local/lib \
  && ./m.o


Install complete
Hello fron GMP
